In [ ]:
# Configure SCBFM_ROOT_DIR and optionally SCBFM_FIGURE_DIR before launching Jupyter.
from pathlib import Path
import os
import sys

_candidates = [Path(os.environ['SCBFM_REPO_DIR'])] if os.environ.get('SCBFM_REPO_DIR') else []
_candidates += [Path.cwd(), *Path.cwd().parents]
REPO_ROOT = next((p for p in _candidates if (p / 'src' / 'main.py').is_file()), None)
if REPO_ROOT is None:
    raise FileNotFoundError('Open Jupyter inside the checkout or set SCBFM_REPO_DIR.')
if str(REPO_ROOT / 'src') not in sys.path:
    sys.path.insert(0, str(REPO_ROOT / 'src'))
from notebook_setup import ROOT_DIR, OUTPUT_DIR, FIGURE_DIR


In [ ]:
import re
import numpy as np
import pandas as pd
import anndata as ad
from scipy import sparse

## Part I - Import and reorganization to homogene h5ad structure

In [ ]:
lincs = ad.read_h5ad(str(ROOT_DIR / 'datasets/LINCS/compound_perturbation_data.h5ad'))

In [ ]:
lincs.var["symbol"] = lincs.var_names.copy()
lincs.var_names = lincs.var["ensg_id"].astype(str).copy()
lincs.var = lincs.var.drop(columns=["ensg_id"])
lincs.var.index.name = "ensg_id" 

In [ ]:
# remove nan genes
valid_gene_mask = ~pd.isna(lincs.var_names)
valid_gene_mask &= lincs.var_names.astype(str) != "nan"

lincs = lincs[:, valid_gene_mask].copy()

In [ ]:
lincs.write_h5ad(str(ROOT_DIR / 'datasets/LINCS/lincs.h5ad'))

## Part II - Statistics

In [ ]:
with open(str(REPO_ROOT / 'data/gene_list.txt')) as f:
    gene_list = [line.strip() for line in f if line.strip()]

In [ ]:
lincs = ad.read_h5ad(str(ROOT_DIR / 'datasets/LINCS/lincs.h5ad'))
print(lincs)

In [ ]:
gene_set = set(map(str, gene_list))
var_names = np.asarray(lincs.var_names.astype(str))

in_list_mask = np.array([g in gene_set for g in var_names], dtype=bool)
not_in_list_mask = ~in_list_mask
not_in_list_weights = not_in_list_mask.astype(np.float64)

chunk_size = 1000

sum_frac_nonzero = 0.0
sum_frac_reads = 0.0
n_obs_done = 0

for start in range(0, lincs.n_obs, chunk_size):
    end = min(start + chunk_size, lincs.n_obs)

    X_chunk = lincs.X[start:end]

    if sparse.issparse(X_chunk):
        X_chunk = X_chunk.tocsr()

        total_nonzero = np.asarray(X_chunk.getnnz(axis=1)).ravel()
        total_reads = np.asarray(X_chunk.sum(axis=1)).ravel()

        # Same as X_chunk[:, not_in_list_mask].sum(axis=1), but avoids sparse fancy indexing.
        not_in_list_reads = np.asarray(X_chunk @ not_in_list_weights).ravel()

        X_binary = X_chunk.copy()
        X_binary.data = np.ones_like(X_binary.data, dtype=np.float64)
        not_in_list_nonzero = np.asarray(X_binary @ not_in_list_weights).ravel()

        del X_binary

    else:
        X_chunk = np.asarray(X_chunk)

        total_nonzero = (X_chunk > 0).sum(axis=1)
        not_in_list_nonzero = (X_chunk[:, not_in_list_mask] > 0).sum(axis=1)

        total_reads = X_chunk.sum(axis=1)
        not_in_list_reads = X_chunk[:, not_in_list_mask].sum(axis=1)

    frac_nonzero_not_in_list = np.divide(
        not_in_list_nonzero,
        total_nonzero,
        out=np.zeros_like(total_nonzero, dtype=float),
        where=total_nonzero > 0,
    )

    frac_reads_not_in_list = np.divide(
        not_in_list_reads,
        total_reads,
        out=np.zeros_like(total_reads, dtype=float),
        where=total_reads > 0,
    )

    sum_frac_nonzero += frac_nonzero_not_in_list.sum()
    sum_frac_reads += frac_reads_not_in_list.sum()
    n_obs_done += end - start

    if start == 0 or n_obs_done % (10 * chunk_size) == 0 or end == lincs.n_obs:
        print(f"Processed {n_obs_done:,}/{lincs.n_obs:,} samples")

    del X_chunk

print("Average portion of non-zero genes NOT in gene_list:",
      sum_frac_nonzero / n_obs_done)

print("Average portion of total reads NOT in gene_list:",
      sum_frac_reads / n_obs_done)


## Part III - filter to gene list

In [ ]:
gene_list = [str(g) for g in gene_list]
gene_index = pd.Index(lincs.var_names.astype(str))

present_genes = [g for g in gene_list if g in gene_index]
missing = [g for g in gene_list if g not in gene_index]
present_src_idx = gene_index.get_indexer(present_genes)

print(f"LINCS genes present: {len(present_genes)} / {len(gene_list)}")
print(f"LINCS genes missing: {len(missing)}")

out_path = str(ROOT_DIR / 'datasets/LINCS/lincs.h5ad')
chunk_size = 1000
chunks = []

for start in range(0, lincs.n_obs, chunk_size):
    end = min(start + chunk_size, lincs.n_obs)
    X_chunk = lincs.X[start:end, :]

    if sparse.issparse(X_chunk):
        # CSC is safer for column selection on some SciPy builds.
        X_out = X_chunk.tocsc()[:, present_src_idx].tocsr()
    else:
        X_out = np.asarray(X_chunk)[:, present_src_idx]

    chunk = ad.AnnData(
        X=X_out,
        obs=lincs.obs.iloc[start:end].copy(),
        var=pd.DataFrame(
            index=pd.Index(present_genes, name=lincs.var_names.name),
        ),
    )
    chunks.append(chunk)

    print(f"Prepared {end:,}/{lincs.n_obs:,} samples")

lincs_filtered = ad.concat(chunks, axis=0, join="inner", merge="same")
lincs_filtered.var_names = pd.Index(present_genes, name=lincs.var_names.name)

lincs_filtered.write_h5ad(out_path)
print("Wrote:", out_path)
print("Shape:", lincs_filtered.shape)
